# Lecture 6.6: Hands-On: Multi-Turn Code Analysis Agent

**Section 06: Sessions and Context Management (capstone)**

This notebook teaches nothing new. It combines every idea from the section in one small agent workflow that interrogates a Python project over several turns, inside one session.

By the end you will have an agent that:

* **keeps its memory** across turns by resuming the same session,
* **applies a fix with file checkpointing on**, so the edit can be undone,
* **branches the session** to explore an alternative without touching the main thread,
* **reports its context size and estimated cost**.

The project under investigation is a small Seed Library app. **It contains one bug for the agent to find.** You will not be told where it is.

### Cell map

| Stage | Cells | What happens |
| :-- | :-- | :-- |
| Setup | 1 to 5 | Install, API key, model, create the project, run a ground-truth check |
| Helper | 6 to 7 | Build the `make_options()` and `ask()` helpers every turn will use |
| Analyse | 8 to 10 | Turn 1 reads the code. Turn 2 builds on turn 1 without re-reading. |
| Fix and rewind | 11 to 15 | Turn 3 fixes the bug with checkpointing on. The edit is undone. Turn 4 fixes it a different way. |
| Branch | 16 to 17 | Fork the session for a read-only design proposal |
| Report | 18 | Context size, estimated session cost, session list, file state |

Every code cell has a number in its heading and in its first comment line, so you can always find your place.

## Cell 1: Install the Claude Agent SDK

This cell installs the only package this notebook needs, `claude-agent-sdk`. The package ships with the Claude Code command line tool bundled inside it, which is the engine that actually runs the agent loop. Installing the package is therefore all the setup the agent needs. Nothing else has to be installed on your machine.

Two details are worth noting:

* **The version is pinned** (`==0.2.167`) so that every example behaves exactly as shown on screen. Newer versions bundle a newer engine, and behaviour can shift between releases.
* **`-q` keeps the output quiet.** If the package is already present in your Colab session, pip simply confirms it and finishes quickly.

Run the cell and wait for it to finish before moving on.

In [ ]:
# Cell 1: Install the Claude Agent SDK

# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.167 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.7 MB/s eta 0:00:00


## Cell 2: Load your API key from Colab Secrets

The agent needs an Anthropic API key to call the model. This notebook reads the key from **Colab Secrets**, so it never appears in the notebook itself.

**One-time setup in Colab:**

1. Create a key in the Claude Console at [platform.claude.com](https://platform.claude.com/) under **API Keys**, and copy it.
2. In Colab, click the **key icon** in the left sidebar.
3. Click **Add new secret**.
4. Name it exactly `ANTHROPIC_API_KEY` and paste the key as the value.
5. Switch **Notebook access** on for this notebook.

The cell then reads the secret with `userdata.get(...)` and writes it into the `ANTHROPIC_API_KEY` environment variable, which is where the SDK looks for it.

> **Running locally instead of Colab?** Skip this cell and set the environment variable in your terminal before starting Jupyter, for example `export ANTHROPIC_API_KEY="your-key"`.

In [ ]:
# Cell 2: API key from Colab Secrets
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

This cell stores the model name in one variable, `MODEL_NAME`. Every agent call in this notebook passes `model=MODEL_NAME`, so **changing this one line changes the model for the entire notebook**.

The notebook uses Haiku 5.5, a fast model that keeps a multi-turn demo quick and inexpensive. Model names change with releases, so the comment in the cell links to the models page where you can find the latest available names.

In [ ]:
# Cell 3: Model name used everywhere in this notebook
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-5-5"

## Cell 4: Create the Seed Library project

The agent needs something to investigate, so this cell creates a small project under `/content/seedlibrary_app/`:

| File | Purpose |
| :-- | :-- |
| `catalog.txt` | Six lines of seed stock, one variety per line |
| `inventory.py` | Functions to parse a catalog line, load the catalog, and find varieties that are running low |
| `report.py` | Imports from `inventory` and prints a short stock summary |

**The project contains one bug for the agent to find.** Nothing in this notebook tells you where it is.

The cell also records two things that make the rest of the notebook trustworthy:

* **`ORIGINAL_HASHES`**: a SHA-256 fingerprint of each file. If a file changes, its fingerprint changes. This is how we will check what the agent really did, instead of believing what it says.
* **`ORIGINAL_INVENTORY`**: the original text of `inventory.py`, kept in a variable so that a later cell can print an exact line-by-line diff of any change.

Re-running this cell rewrites the files from scratch, which resets the project to its original state.

In [ ]:
# Cell 4: Create the Seed Library project and record the original state
from pathlib import Path
import hashlib

APP_DIR = Path("/content/seedlibrary_app")
APP_DIR.mkdir(parents=True, exist_ok=True)

CATALOG_LINES = [
    "marigold: 40 packets",
    "basil: 12 packets",
    "sunflower: 55 packets",
    "zinnia: 8 packets",
    "nasturtium: 30 packets",
    "cosmos: 18 packets",
]
TOTAL_LINES = len(CATALOG_LINES)

ORIGINAL_INVENTORY = """def parse_line(line):
    name, rest = line.split(":", 1)
    count = int(rest.split()[0])
    return name.strip(), count


def load_catalog(path):
    with open(path) as handle:
        lines = handle.read().splitlines()

    catalog = {}
    for line in lines[:-1]:
        if line.strip():
            name, count = parse_line(line)
            catalog[name] = count
    return catalog


def low_stock(catalog, threshold):
    return sorted(name for name, count in catalog.items() if count < threshold)
"""

REPORT_PY = """from inventory import load_catalog, low_stock

THRESHOLD = 25

catalog = load_catalog("catalog.txt")
print(f"Seed Library stock report: {len(catalog)} varieties")
for name, count in sorted(catalog.items()):
    print(f"  {name}: {count} packets")

running_low = low_stock(catalog, THRESHOLD)
print("Running low: " + ", ".join(running_low))
"""

(APP_DIR / "catalog.txt").write_text("\n".join(CATALOG_LINES) + "\n")
(APP_DIR / "inventory.py").write_text(ORIGINAL_INVENTORY)
(APP_DIR / "report.py").write_text(REPORT_PY)


def file_hashes():
    return {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(APP_DIR.glob("*"))
        if path.is_file()
    }


ORIGINAL_HASHES = file_hashes()
print(f"Created {len(ORIGINAL_HASHES)} files in {APP_DIR}")

Created 3 files in /content/seedlibrary_app


## Cell 5: Verify the project and define the ground-truth tools

Before any agent runs, we look at exactly what exists. This cell prints the directory tree and the contents of every file, so there are no surprises later.

It then defines three small tools that the rest of the notebook relies on. **They look at the filesystem, not at what the agent says.**

| Tool | What it does |
| :-- | :-- |
| `check_catalog()` | Runs the app's `load_catalog` in a **fresh subprocess** and prints `loaded N of 6 lines`. A subprocess is used instead of an import because a notebook keeps imported modules cached. An import would keep showing the old code even after a file changed on disk. |
| `show_file_state(baseline)` | Compares each file's current fingerprint with a baseline and marks it `same` or `CHANGED`. |
| `show_diff(label)` | Prints a unified diff between the original `inventory.py` text and the file as it is now. |

The cell finishes by running `check_catalog()`. Treat that number as your **baseline**. The project has one bug, and this is its symptom. Where it comes from is the agent's job to find.

In [ ]:
# Cell 5: Verify the project and define the ground-truth tools
import difflib
import os
import subprocess
import sys

print("Directory tree:")
for path in sorted(APP_DIR.rglob("*")):
    if "__pycache__" in path.parts:
        continue
    depth = len(path.relative_to(APP_DIR).parts)
    print("  " * depth + path.name)

for path in sorted(APP_DIR.glob("*")):
    if path.is_file():
        print()
        print(f"===== {path.name} =====")
        print(path.read_text())


def check_catalog():
    probe = """
from inventory import load_catalog
print(len(load_catalog("catalog.txt")))
"""
    completed = subprocess.run(
        [sys.executable, "-c", probe],
        cwd=APP_DIR,
        capture_output=True,
        text=True,
        env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},
    )
    if completed.returncode != 0:
        print("check_catalog failed:")
        print(completed.stderr)
        return None
    loaded = int(completed.stdout.strip())
    print(f"loaded {loaded} of {TOTAL_LINES} lines")
    return loaded


def show_file_state(baseline):
    current = file_hashes()
    for name in sorted(current):
        status = "same" if current[name] == baseline.get(name) else "CHANGED"
        print(f"  {name:<14}{current[name][:12]}  {status}")
    return current == baseline


def show_diff(label):
    current = (APP_DIR / "inventory.py").read_text()
    diff = difflib.unified_diff(
        ORIGINAL_INVENTORY.splitlines(keepends=True),
        current.splitlines(keepends=True),
        fromfile="inventory.py (original)",
        tofile=f"inventory.py ({label})",
    )
    text = "".join(diff)
    print(text if text else f"No difference from the original ({label}).")


print()
print("Baseline check:")
check_catalog()

Directory tree:
  catalog.txt
  inventory.py
  report.py

===== catalog.txt =====
marigold: 40 packets
basil: 12 packets
sunflower: 55 packets
zinnia: 8 packets
nasturtium: 30 packets
cosmos: 18 packets


===== inventory.py =====
def parse_line(line):
    name, rest = line.split(":", 1)
    count = int(rest.split()[0])
    return name.strip(), count


def load_catalog(path):
    with open(path) as handle:
        lines = handle.read().splitlines()

    catalog = {}
    for line in lines[:-1]:
        if line.strip():
            name, count = parse_line(line)
            catalog[name] = count
    return catalog


def low_stock(catalog, threshold):
    return sorted(name for name, count in catalog.items() if count < threshold)


===== report.py =====
from inventory import load_catalog, low_stock

THRESHOLD = 25

catalog = load_catalog("catalog.txt")
print(f"Seed Library stock report: {len(catalog)} varieties")
for name, count in sorted(catalog.items()):
    print(f"  {name}: {count} pac

5

## Cell 6: Build `make_options()`

Every turn of this workflow runs through one configuration function, so that all turns share the same settings. `make_options(tools)` returns a `ClaudeAgentOptions` object, and the only thing that changes between turns is the list of tools.

First, a global variable `SESSION_ID` starts as `None`. After the first turn we store the session ID there, and every later turn passes it as `resume`. That single line of plumbing is what gives the agent memory.

| Option | Value | Why |
| :-- | :-- | :-- |
| `resume` | `SESSION_ID` | `None` on the first turn starts a new session. After that, it reopens the same conversation with all earlier context. |
| `enable_file_checkpointing` | `True` | Makes the SDK back up a file before the agent edits it, so the edit can be rewound. It must be on in the session that makes the edit, and in the session that does the rewinding. Setting it here covers every turn. |
| `extra_args` | `{"replay-user-messages": None}` | Required to receive user message IDs in the stream. Those IDs are the checkpoint IDs. `None` means the flag has no value. |
| `allowed_tools` | `tools` | Least privilege. Each turn passes only the tools it needs. |
| `model` | `MODEL_NAME` | The model from Cell 3. |
| `cwd` | `"/content"` | The agent's working directory. |

Checkpointing tracks changes made through the `Write`, `Edit`, and `NotebookEdit` tools only. Changes made through shell commands are not tracked. This notebook only ever edits through `Edit`.

In [ ]:
# Cell 6: One options builder shared by every turn
from claude_agent_sdk import ClaudeAgentOptions

SESSION_ID = None


def make_options(tools):
    return ClaudeAgentOptions(
        resume=SESSION_ID,
        enable_file_checkpointing=True,
        extra_args={"replay-user-messages": None},
        allowed_tools=tools,
        model=MODEL_NAME,
        cwd="/content",
    )

## Cell 7: The `ask()` helper

Each turn of the workflow is one call to `ask(prompt, tools)`. This cell defines it. The helper is pre-built because it is long and mostly plumbing, but the ideas in it are the whole section, so here is the walkthrough in order.

1. **Open a new `ClaudeSDKClient`** using `make_options(tools)`. A client is one connection to the agent. We open a fresh one per turn, and the `resume` option reconnects it to the saved session. The memory lives in the session, not in the client object.
2. **Send the prompt** with `await client.query(prompt)`.
3. **Read the stream** with `client.receive_response()`. Remember the mental model: the agent loop runs on its own and sends back a *stream of messages*. The helper watches for two kinds:
    * the **first `UserMessage` that has a `uuid`**. With `replay-user-messages` on, that ID is this turn's **checkpoint ID**.
    * the final **`ResultMessage`**, which carries the answer in `result`, the `session_id`, and the estimated cost in `total_cost_usd`.
4. **Ask for context usage** with `await client.get_context_usage()` *before leaving the `async with` block*, because the connection closes when the block ends. The response has `totalTokens`, `maxTokens`, and `percentage`.
5. **Update `SESSION_ID`** from the result. We store whatever session ID the latest result reports, so the helper stays correct whether or not that ID changes between turns.
6. **Print** the answer, the context size, and the session's estimated cost, then **return the checkpoint ID**.

**About the cost figure.** `total_cost_usd` is a client-side estimate computed by the SDK. It is not billing data, and the Usage page in the Claude Console is the authoritative record. When a call resumes a session, its result already includes the session's earlier spend. So we always read the **latest** result and never add results together. The field can be `None`, so the helper checks for that.

In [ ]:
# Cell 7: The ask() helper used by every turn
from claude_agent_sdk import ClaudeSDKClient, UserMessage, ResultMessage

LAST_CONTEXT = None
LAST_COST = None


async def ask(prompt, tools):
    global SESSION_ID, LAST_CONTEXT, LAST_COST

    checkpoint_id = None
    result = None

    async with ClaudeSDKClient(make_options(tools)) as client:
        await client.query(prompt)

        async for message in client.receive_response():
            if isinstance(message, UserMessage) and message.uuid and checkpoint_id is None:
                checkpoint_id = message.uuid
            if isinstance(message, ResultMessage):
                result = message

        usage = await client.get_context_usage()

    if result is None:
        raise RuntimeError("The stream ended without a ResultMessage.")

    SESSION_ID = result.session_id
    LAST_CONTEXT = usage
    LAST_COST = result.total_cost_usd

    if result.result:
        print(result.result)
    else:
        print(f"(no result text, subtype: {result.subtype})")

    print()
    print("-" * 60)
    print(f"Session: {SESSION_ID}")
    print(
        f"Context size: {usage['totalTokens']:,} of {usage['maxTokens']:,} tokens "
        f"({usage['percentage']:.1f}%)"
    )
    if result.total_cost_usd is not None:
        print(f"Estimated session cost so far: ${result.total_cost_usd:.4f}")
    else:
        print("Estimated session cost so far: not reported")

    return checkpoint_id

## Cell 8: Turn 1, analyse the project

Time for the first turn. The prompt is a **goal**, not a script: we say what we want and let the agent decide how to get there. It reads the project, explains what it does, and names the function it thinks is most likely to contain a bug.

We pass only read-only tools: `Read`, `Glob`, and `Grep`. The agent can look but cannot change anything.

`SESSION_ID` is `None` here, so this call **starts a new session**. When it finishes, `ask()` stores the new session ID, and every later turn resumes it.

**What to look for.** This is the first moment the project's bug is named, and it is the agent that names it. Live agent output varies from run to run. The wording will differ from what you see on screen, and that is normal behaviour for a real agent, not a problem. Check these three things instead:

* Did the agent read the Python files before answering?
* Did it name a specific function, with a reason?
* Does the output end with a session ID, a context size, and an estimated cost?

In [ ]:
# Cell 8: Turn 1, analyse (read-only tools)
_ = await ask(
    "Read the Python files in /content/seedlibrary_app, explain in a few "
    "sentences what the project does, and name the function most likely to "
    "contain a bug and why.",
    ["Read", "Glob", "Grep"],
)

The project is a small Python stock report for a seed library. `inventory.py` reads a `catalog.txt` file where each line is a variety name followed by a packet count (`Name: 12 packets`). It builds a name-to-count dictionary and lists the varieties below a threshold. `report.py` prints every variety with its count, then a "Running low" line for anything under 25 packets.

The function most likely to contain a bug is `load_catalog` in `inventory.py`. The loop runs over `lines[:-1]`, so it always skips the last line of the file. Unless `catalog.txt` ends with a footer line on purpose, the last variety is silently dropped from the report and from the low-stock check. That error would be easy to miss, since the output still looks plausible. `parse_line` could also raise on a malformed line, but that would fail loudly. I haven't run this against a real `catalog.txt`, and none was in the directory I searched, so the diagnosis is from reading the code.

---------------------------------------

## Cell 9: Turn 2, test the agent's memory

This turn is the proof that resume works. The prompt asks for the exact line of the bug and a one-sentence explanation, and it tells the agent **not to read any files again**.

`ask()` opens a brand new client, but `make_options()` now passes the session ID from turn 1 as `resume`. The session holds the whole earlier conversation: the prompt, every file the agent read, and its own analysis.

We pass an empty tool list. The agent has no reason to touch anything, so it gets nothing extra.

**What to look for.** The answer should build directly on turn 1, quoting the specific line, with no new file reads. Compare the context size with turn 1: it grows, because the conversation so far is carried along with every turn. Context is the cost of memory.

In [ ]:
# Cell 9: Turn 2, answer from memory (no tools, no re-reading)
_ = await ask(
    "Without reading any files again, show me the exact line of code that "
    "contains the bug, and explain the bug in one sentence.",
    [],
)

`inventory.py`, line 12: `    for line in lines[:-1]:`

This slice drops the file's last line on every load, so the final variety is silently left out of the catalog and the low-stock check.

------------------------------------------------------------
Session: 7faa3eb4-5ac1-4bfb-b148-aa8ba6215c6c
Context size: 17,256 of 1,000,000 tokens (2.0%)
Estimated session cost so far: $0.0032


## Cell 10: Confirm nothing changed

Two turns of analysis should have changed nothing on disk, because the agent only had read-only tools. We do not take that on trust. This cell compares the file fingerprints with the ones recorded in Cell 4, then runs `check_catalog()` again.

**What to look for.** Every file should say `same`, the comparison should print `True`, and the catalog check should still show the baseline number from Cell 5. The bug is still there. The agent has found it but has not touched it.

In [ ]:
# Cell 10: Ground truth, nothing changed during analysis
print("File state against the original:")
unchanged = show_file_state(ORIGINAL_HASHES)
print(f"All files identical to the original: {unchanged}")
print()
check_catalog()

File state against the original:
  catalog.txt   46bd538553e5  same
  inventory.py  c3457147e5b8  same
  report.py     35f909dd698c  same
All files identical to the original: True

loaded 5 of 6 lines


5

## Cell 11: Turn 3, fix the bug with checkpointing on

Now the agent gets permission to change the code. The prompt asks for the smallest change that fixes the bug. We pass `["Read", "Edit"]`: it may read files and edit them, and nothing else. Because `make_options()` has `enable_file_checkpointing=True`, the SDK backs up any file before `Edit` modifies it.

`ask()` returns the **checkpoint ID** for this turn, the ID of the first user message in the stream. We store it in `CHECKPOINT_ID`. This is the restore point that rewinds the files to how they were before this turn's edit.

**What to look for.** The agent will report that it fixed the bug. Do not rely on that claim. The next cell checks the filesystem.

In [ ]:
# Cell 11: Turn 3, fix with checkpointing on
CHECKPOINT_ID = await ask(
    "Fix the bug in the project with the smallest change that fixes it. "
    "Do not change anything else.",
    ["Read", "Edit"],
)
print()
print(f"Checkpoint ID for this turn: {CHECKPOINT_ID}")

I changed line 12 of `inventory.py` from `for line in lines[:-1]:` to `for line in lines:`, so the loop now reads every line, including the last one. Nothing else was touched. I haven't run the script, and the fix assumes `catalog.txt` has no footer line that should be skipped.

------------------------------------------------------------
Session: 7faa3eb4-5ac1-4bfb-b148-aa8ba6215c6c
Context size: 17,648 of 1,000,000 tokens (2.0%)
Estimated session cost so far: $0.0038

Checkpoint ID for this turn: 1f1a7c43-d1cf-4c98-af2e-031d87435c87


## Cell 12: Ground truth, did the fix work?

An agent saying it fixed a bug is a claim. A file that changed and a check that passes are facts. This cell gathers three pieces of evidence:

1. `check_catalog()` runs the app in a fresh subprocess and reports how many lines loaded.
2. `show_file_state()` shows which files' fingerprints changed.
3. `show_diff()` prints a unified diff between the original `inventory.py` and the file now. Lines starting with `-` were removed and lines starting with `+` were added.

**What to look for.** The catalog check should now report all six lines, only the file the agent was supposed to edit should show `CHANGED`, and the diff should be small. **Trust the filesystem and the check, not the agent's description of its own work.**

In [ ]:
# Cell 12: Ground truth after the fix
check_catalog()
print()
print("File state against the original:")
show_file_state(ORIGINAL_HASHES)
print()
show_diff("after turn 3")

loaded 6 of 6 lines

File state against the original:
  catalog.txt   46bd538553e5  same
  inventory.py  139d82075b77  CHANGED
  report.py     35f909dd698c  same

--- inventory.py (original)
+++ inventory.py (after turn 3)
@@ -9,7 +9,7 @@
         lines = handle.read().splitlines()
 
     catalog = {}
-    for line in lines[:-1]:
+    for line in lines:
         if line.strip():
             name, count = parse_line(line)
             catalog[name] = count



## Cell 13: Rewind the edit

Now we undo the change. File rewinding restores files on disk to how they were at a checkpoint. It works after the original stream has finished by **resuming the session with an empty prompt** and calling `rewind_files()`:

1. Open a client with `make_options([])`. It resumes `SESSION_ID`, and checkpointing is on, which the resumed session needs in order to rewind.
2. Send an empty prompt. This only opens the connection. No new work is requested.
3. Inside the response loop, call `await client.rewind_files(CHECKPOINT_ID)` and then `break`. The call has to happen while the connection is open. Calling it after the loop finishes fails with a "not ready for writing" error.

`rewind_files()` restores tracked files only, meaning files changed through `Write`, `Edit`, or `NotebookEdit`. It does not touch the conversation.

In [ ]:
# Cell 13: Rewind the files to the checkpoint
async with ClaudeSDKClient(make_options([])) as client:
    await client.query("")
    async for message in client.receive_response():
        await client.rewind_files(CHECKPOINT_ID)
        break

print("rewind_files() called with the turn 3 checkpoint ID.")

rewind_files() called with the turn 3 checkpoint ID.


## Cell 14: Ground truth, the files went back

Same method as before: check the filesystem, not the story.

**What to look for.** The fingerprint comparison with the original should print `True`, with every file marked `same`, and the catalog check should be back at the baseline number. The fix is gone.

**This is the lesson of the capstone: the files went back, and the conversation did not.** Rewinding restores files. It does not rewind the session. The agent still remembers making that edit, and the next cell shows why that matters.

In [ ]:
# Cell 14: Ground truth after the rewind
print("File state against the original:")
restored = show_file_state(ORIGINAL_HASHES)
print(f"All files identical to the original: {restored}")
print()
check_catalog()

File state against the original:
  catalog.txt   46bd538553e5  same
  inventory.py  c3457147e5b8  same
  report.py     35f909dd698c  same
All files identical to the original: True

loaded 5 of 6 lines


5

## Cell 15: Turn 4, fix it a different way

The session still contains turn 3: the agent's edit and its confident report that the bug was fixed. But the file on disk no longer has that edit. If we just said "fix the bug again", the agent could reasonably believe the work was already done.

So the prompt **tells the agent the file was reverted**. It then asks the agent to do two things: describe what it changed before, which proves the agent still remembers turn 3, and fix the bug a different way.

After the turn, the cell records a snapshot of the file fingerprints in `SNAPSHOT_AFTER_FIX`. Cell 17 uses it to prove the fork did not touch the files. It also runs the same ground-truth checks as before.

**What to look for.** The agent should describe the earlier change accurately, then make a different edit. The catalog check should report all six lines again, and the diff should look different from the one in Cell 12.

In [ ]:
# Cell 15: Turn 4, tell the agent the change was undone, then fix it differently
_ = await ask(
    "The change you made earlier has been undone, so the project files are "
    "back to their original state. First, describe exactly what you changed "
    "before. Then fix the bug a different way, keeping the change small.",
    ["Read", "Edit"],
)

print()
check_catalog()
print()
print("File state against the original:")
show_file_state(ORIGINAL_HASHES)
print()
show_diff("after turn 4")

SNAPSHOT_AFTER_FIX = file_hashes()

The earlier change was on line 12 of `inventory.py`: `for line in lines[:-1]:` became `for line in lines:`. I've confirmed the file was back to its original state before this edit.

This time I changed `load_catalog` so it loops over the open file directly, instead of building a list of lines and slicing off the last one. The `lines` variable is gone, and the loop body is indented one level deeper inside the `with` block. Lines with only whitespace are still skipped, and `parse_line` handles the trailing newline that file iteration keeps. I haven't run the script.

------------------------------------------------------------
Session: 7faa3eb4-5ac1-4bfb-b148-aa8ba6215c6c
Context size: 19,771 of 1,000,000 tokens (2.0%)
Estimated session cost so far: $0.0056

loaded 6 of 6 lines

File state against the original:
  catalog.txt   46bd538553e5  same
  inventory.py  080420be3782  CHANGED
  report.py     35f909dd698c  same

--- inventory.py (original)
+++ inventory.py (after turn 4)
@@ -5,14 +

## Cell 16: Fork the session for a read-only proposal

Forking creates a **new session that starts with a copy of the original's history**. The original session stays unchanged. It is how you explore an alternative without risking the main thread.

This cell uses a single `query()` call with two options doing the work:

| Option | Meaning |
| :-- | :-- |
| `resume=SESSION_ID` | Start from the main session's history |
| `fork_session=True` | Branch into a new session instead of continuing the original |

The fork's job is read-only. We ask for a design proposal in prose and forbid edits, and we pass only `["Read"]` as allowed tools. The fork's ID is read from its `ResultMessage` and stored in `FORK_ID`.

**Two design choices to notice.** First, we keep checkpointing and forking apart: checkpointing stays on the main thread, and the fork never edits. Forking branches the conversation, not the filesystem, so if a fork edited a file, that change would be real. Second, we do not update `SESSION_ID` from the fork. The main thread keeps its own ID.

**What to look for.** A thoughtful answer in prose about separating parsing from reporting, and a fork session ID printed at the end.

In [ ]:
# Cell 16: Fork the main session for a read-only redesign proposal
from claude_agent_sdk import query

FORK_ID = None

async for message in query(
    prompt=(
        "In prose only, with no edits and no file changes, explain how this "
        "project could separate parsing from reporting."
    ),
    options=ClaudeAgentOptions(
        resume=SESSION_ID,
        fork_session=True,
        allowed_tools=["Read"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage):
        FORK_ID = message.session_id
        if message.result:
            print(message.result)

print()
print("-" * 60)
print(f"Fork session: {FORK_ID}")

The project already splits the work in two, but the boundary is loose. `inventory.py` handles reading and parsing, and it also decides which varieties are running low. `report.py` loads the catalog, sets the threshold, and does all the printing, all at module level.

A cleaner split would give each piece one job:

- **Parsing** would turn text into plain data. `parse_line` and `load_catalog` would stay in `inventory.py`, but `load_catalog` could accept an iterable of lines or a string rather than a file path. That way it can be tested without touching the disk. Its only output would be a dictionary of names to counts.
- **Analysis** would be pure functions over that dictionary, such as `low_stock`. These would take the catalog and threshold as arguments and return results without printing or reading files.
- **Reporting** would live in `report.py` and would only format and print. It would take the catalog and the low-stock list as inputs, and it would not know how parsing works.
- **Wi

## Cell 17: Independence and ground truth

We now prove that the fork really is separate. The cell checks four things:

1. **The fork has its own ID.** `FORK_ID` should differ from `SESSION_ID`.
2. **The project files did not change.** Their fingerprints should match the snapshot taken after turn 4.
3. **The catalog check still passes.** It should report all six lines.
4. **The sessions are listed.** `list_sessions(directory="/content")` returns the sessions saved for this working directory, newest first. The cell labels each one `main` or `fork`.

**What to look for.** The main thread is untouched and the fork is its own session. The number of sessions listed may be larger than two. Anything labelled `other` is a session the SDK saved for this directory during the notebook, and the labels tell you which two are yours.

In [ ]:
# Cell 17: The fork is independent, the main thread is untouched
from claude_agent_sdk import list_sessions

print(f"Main session: {SESSION_ID}")
print(f"Fork session: {FORK_ID}")
print(f"Different IDs: {FORK_ID != SESSION_ID}")
print()

print("File state against the snapshot taken after turn 4:")
unchanged = show_file_state(SNAPSHOT_AFTER_FIX)
print(f"Files unchanged since turn 4: {unchanged}")
print()
check_catalog()
print()

sessions = list_sessions(directory="/content")
print(f"Sessions listed for /content: {len(sessions)}")
for session in sessions:
    if session.session_id == SESSION_ID:
        label = "main"
    elif session.session_id == FORK_ID:
        label = "fork"
    else:
        label = "other"
    print(f"  {label:<6}{session.session_id}  {session.summary[:50]}")

Main session: 7faa3eb4-5ac1-4bfb-b148-aa8ba6215c6c
Fork session: ac3e154d-453f-4495-ad7f-da04e777d4b5
Different IDs: True

File state against the snapshot taken after turn 4:
  catalog.txt   46bd538553e5  same
  inventory.py  080420be3782  same
  report.py     35f909dd698c  same
Files unchanged since turn 4: True

loaded 6 of 6 lines

Sessions listed for /content: 2
  fork  ac3e154d-453f-4495-ad7f-da04e777d4b5  Seedlibrary_app Python project overview
  main  7faa3eb4-5ac1-4bfb-b148-aa8ba6215c6c  Seedlibrary_app Python project overview


## Cell 18: Final report

The last cell gathers the meters into one table: how many sessions exist, how large the main session's context is, what it has cost so far, and the state of the files. It reads values already collected, plus one fresh `list_sessions()` call and one fresh `check_catalog()`.

The context size is from the last turn of the main session. The cost is the **session's estimated cost**, taken from the latest result and never summed. It is a client-side estimate, not billing data.

In [ ]:
# Cell 18: Final report
from IPython.display import Markdown, display

sessions = list_sessions(directory="/content")
loaded = check_catalog()
changed = file_hashes() != ORIGINAL_HASHES

context_text = (
    f"{LAST_CONTEXT['totalTokens']:,} of {LAST_CONTEXT['maxTokens']:,} tokens "
    f"({LAST_CONTEXT['percentage']:.1f}%)"
)
cost_text = f"${LAST_COST:.4f}" if LAST_COST is not None else "not reported"
files_text = (
    f"inventory.py differs from the original: {'yes' if changed else 'no'}; "
    f"catalog check: loaded {loaded} of {TOTAL_LINES} lines"
)

display(
    Markdown(
        "| Meter | Value |\n"
        "| :-- | :-- |\n"
        f"| Sessions listed for /content | {len(sessions)} |\n"
        f"| Last context size (main session) | {context_text} |\n"
        f"| Estimated session cost | {cost_text} |\n"
        f"| File state | {files_text} |\n"
    )
)

loaded 6 of 6 lines


| Meter | Value |
| :-- | :-- |
| Sessions listed for /content | 2 |
| Last context size (main session) | 19,771 of 1,000,000 tokens (2.0%) |
| Estimated session cost | $0.0056 |
| File state | inventory.py differs from the original: yes; catalog check: loaded 6 of 6 lines |


## Wrap-up: what you built

One agent workflow, four ideas from the section, each doing a different job:

| Stage | What happened | Section idea it drew on |
| :-- | :-- | :-- |
| Setup | Created the project, recorded fingerprints, ran a ground-truth check | Lecture 6.1: sessions live on disk and can be listed |
| Analyse | Turn 1 read the code. Turn 2 answered from memory. | Lecture 6.2: resume gives the agent its memory |
| Fix | Turn 3 edited the code with checkpointing on | Lecture 6.4: checkpointing |
| Rewind | The files went back. The conversation did not. Turn 4 fixed it differently. | Lecture 6.4: rewind restores files, not memory |
| Branch | A fork explored a redesign without touching the main thread | Lecture 6.3: forking |
| Report | Context size, estimated session cost, session list | Lecture 6.5 and Lecture 6.1 |

The takeaway: **a session is the memory, checkpoints are the undo for files, forks are the branch, and context and cost are the meters.**

### Try it yourself

* Run the notebook two or three times from Cell 4. Notice how the agent's wording varies while the ground-truth checks stay the same.
* Change `MODEL_NAME` in Cell 3 and compare how the agent behaves. Find current model names on the [models page](https://platform.claude.com/docs/en/about-claude/models/overview).